# Math LLM Probing & Evaluation Pipeline

This notebook implements a pipeline to:
1. Load `Qwen/Qwen2.5-Math-7B-Instruct`.
2. Generate answers to math problems (enforcing direct answers).
3. Extract hidden states.
4. Evaluate answers using Regex and SymPy.

In [ ]:
# Install dependencies if running in Colab
!pip install transformers torch sympy accelerate

In [ ]:
import sys
import os

# Add the current directory to path to import src
sys.path.append(os.getcwd())

try:
    from src import model_utils, evaluator
except ImportError:
    # If running in Colab and files are not uploaded, you might need to clone the repo or upload src/
    print("Make sure 'src' directory is present.")

In [ ]:
# Load Model
model, tokenizer = model_utils.load_model(model_name="Qwen/Qwen2.5-Math-7B-Instruct")

In [ ]:
# Define some test questions (or load from a dataset)
questions = [
    {"question": "What is 2 + 2?", "answer": "4"},
    {"question": "Solve for x: 2x + 5 = 15", "answer": "5"},
    {"question": "Calculate the derivative of x^2", "answer": "2*x"}
]

In [ ]:
# Run Pipeline
results = []

for item in questions:
    q = item["question"]
    gt = item["answer"]
    
    print(f"Processing: {q}")
    
    # Generate
    pred_text, hidden_states = model_utils.generate_answer(model, tokenizer, q)
    
    # Extract clean answer
    clean_pred = evaluator.extract_answer(pred_text)
    
    # Evaluate
    is_correct = evaluator.is_equivalent(clean_pred, gt)
    
    print(f"  Pred: {pred_text}")
    print(f"  Clean: {clean_pred}")
    print(f"  Correct: {is_correct}")
    
    results.append({
        "question": q,
        "ground_truth": gt,
        "prediction": pred_text,
        "clean_prediction": clean_pred,
        "correct": is_correct,
        # Store hidden states if needed, or save to disk to save RAM
        # "hidden_states": hidden_states 
    })

In [ ]:
# Analyze Results
correct_count = sum(1 for r in results if r['correct'])
accuracy = correct_count / len(results)
print(f"Accuracy: {accuracy:.2%}")